# Unemployment-rate ranking CIs

Same procedure as `NGDP_CI`, but for the SPF unemployment-rate panel:
- **Forecasts**: `SPFmicrodata.xlsx`, sheet `UNEMP` (UNEMP1..UNEMP6).
- **Realizations**: `rucQvMd.xlsx` — monthly RTDSM vintage matrix; we average each quarter's three months to get the quarterly UR.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from rankci import (
    rank_ci_stepwise_pairwise,
    compute_pairwise,
    load_spf,
    load_rtdsm,
    compute_errors,
    compute_squared_error_panel,
    select_top_forecasters,
    winsorize_panel,
)
from rankci.data import advance_vintage_col, get_advance_estimate

# Data Loading

In [ ]:
df = load_spf("../data/SPFmicrodata.xlsx", sheet="UNEMP")
print(f"SPF UNEMP: {df.shape[0]} rows × {df.shape[1]} columns")
print("Columns:", df.columns.tolist())
df.head()

In [ ]:
# Monthly RTDSM, averaged into quarterly observations under the hood
ruc = load_rtdsm("../data/rucQvMd.xlsx", prefix="RUC", freq="monthly")
print(f"RTDSM (RUC): {ruc.shape[0]} quarters × {ruc.shape[1]} vintages")
ruc.head()

# Sanity Checks

In [ ]:
# Verify advance UR estimates for a few known quarters
test_cases = [(1995, 2), (2008, 4), (2020, 2)]
print(f"{'Target Quarter':<20} {'Vintage Col':<14} {'UR (%)'}")
print("-" * 50)
for y, q in test_cases:
    col = advance_vintage_col(y, q, prefix="RUC")
    val = get_advance_estimate(y, q, ruc)
    print(f"  {y}:Q{q:<16}  {col:<14}  {val:.2f}")

# Forecast Errors

In [ ]:
errors_df = compute_errors(df, ruc, indicator="UNEMP")

print("Error summary by horizon (percentage points):")
for h in range(1, 7):
    col = f"error_UNEMP{h}"
    if col in errors_df.columns:
        print(f"  {col}: mean={errors_df[col].mean():+.2f}, "
              f"std={errors_df[col].std():.2f}, "
              f"nan%={errors_df[col].isna().mean()*100:.1f}%")

errors_df.head()

In [ ]:
# Squared error panel for UNEMP3 (one-quarter-ahead)
X_wide = compute_squared_error_panel(df, ruc, indicator="UNEMP", horizon=3)
print(f"Panel shape: {X_wide.shape[0]} quarters × {X_wide.shape[1]} forecasters")
X_wide.head()

# Forecaster Selection

In [ ]:
obs_counts = X_wide.notna().sum()
mean_mse = X_wide.mean()

stats = pd.DataFrame({
    "obs_count": obs_counts,
    "mean_mse":  mean_mse,
    "rmse":      np.sqrt(mean_mse),
}).sort_values("obs_count", ascending=False)

print(f"Total forecasters: {len(stats)}")
print(f"Forecasters with >= 20 obs: {(stats['obs_count'] >= 20).sum()}")
print(f"Forecasters with >= 50 obs: {(stats['obs_count'] >= 50).sum()}")
stats.head(20)

In [ ]:
N = 8
X_panel = select_top_forecasters(X_wide, N=N, min_obs=20)
print(f"Selected {X_panel.shape[1]} forecasters, {X_panel.shape[0]} quarters")
print(f"Forecaster IDs: {X_panel.columns.tolist()}")

# Rank Confidence Intervals

In [ ]:
X = X_panel.values
population_ids = X_panel.columns.tolist()

out = rank_ci_stepwise_pairwise(X, alpha=0.05, B=5000, seed=42)

results = pd.DataFrame({
    "ID":         population_ids,
    "MSE":        out["theta_hat"].round(4),
    "RMSE":       np.sqrt(out["theta_hat"]).round(4),
    "CI_lower":   out["rank_ci"][:, 0],
    "CI_upper":   out["rank_ci"][:, 1],
}).sort_values("MSE")
results.index = range(1, len(results) + 1)
results.index.name = "Rank"
results

# Worst-Quarter Inspection

In [ ]:
period_mse = X_panel.mean(axis=1)

print(f"Time coverage: {X_panel.index[0]} to {X_panel.index[-1]} ({len(X_panel)} quarters)\n")
print("Top 10 quarters with highest average squared error:")
print(period_mse.nlargest(10))

n = len(X_panel)
plt.figure(figsize=(12, 4))
plt.plot(range(n), period_mse, marker="o", linewidth=1, markersize=3)
plt.xticks(
    ticks=range(0, n, 8),
    labels=[f"{y}Q{q}" for y, q in X_panel.index[::8]],
    rotation=45,
)
plt.ylabel("Avg squared UR error (pp²)")
plt.title(f"Avg squared UNEMP3 forecast error across {N} forecasters")
plt.tight_layout()
plt.show()

# Winsorization (variant 2: pairwise differences)

In [ ]:
print("=== Winsorize pairwise differences ===")
print(f"{'winsor_pct':<12} {'max_t':<10} {'mean CI width':<16}")
print("-" * 40)
for winsor_pct in [99, 97, 95, 90]:
    delta_w, se_w, _ = compute_pairwise(X, se_method="nw", winsor_pct=winsor_pct)
    t_vals = (delta_w / se_w)[~np.isnan(se_w)]

    out_w = rank_ci_stepwise_pairwise(
        X, alpha=0.05, B=1000, seed=42, winsor_pct=winsor_pct, verbose=False,
    )
    widths = out_w["rank_ci"][:, 1] - out_w["rank_ci"][:, 0]

    print(f"  {winsor_pct:<10} {t_vals.max():<10.3f} {widths.mean():<16.2f}")